In [1]:
import os
import pymupdf
from PIL import Image
import pytesseract
import numpy as np
import re
file_path = os.path.join("..","data","sanskrit_english_bilingual_stories.pdf")
print(file_path)

../data/sanskrit_english_bilingual_stories.pdf


In [2]:
doc = pymupdf.open(file_path)
for page in doc:
    data = page.get_text()
    print(data)

१. ɭसिंहः च मूषकः
एकɧ×स्मिन् वने एकः बलवान् ɭसिंहः वसिɟति ×स्मि। एकदा सिः वृक्ष×य अधः सिुप्तिः आसिीति्। तिदा एकः लघुः स्मिूषकः तित्र आगत्य
ɭसिंह×य शरीरे इति×तितिः अधावति्। तिेन ɭसिंह×य ɟनद्रा भग्ना अभवति्। ɭसिंहः क्रुद्धः भूत्वा स्मिूषकं ह×तिेन अगृह्णाति्। स्मिूषकः
भीतिः भूत्वा अवदति्- 'भो वनराज! स्मिां स्मिुञ्चतिु। कदाɡचति् अहस्मि् अɟपि भवतिः सिाहाय्यं कɝरष्याɠस्मि।' ɭसिंहः हɡसित्वा तिं
अस्मिुञ्चति्। गच्छɟति काले एकदा ɭसिंहः व्याध×य जाले बद्धः अभवति्। सिः उच्चैः अगजर्जति्। ɭसिंह×य गजर्जनं श्रुत्वा स्मिूषकः तित्र
आगच्छति्। सिः ×वकैः तिीक्ष्णैः दन्तिैः जालं अकृन्तिति्। ɭसिंहः जालाति् स्मिुक्तिः अभवति्। सिः स्मिूषक×य धन्यवादं अकरोति्।
1. The Lion and the Mouse
In a forest, there lived a powerful lion. One day, he was sleeping under a tree. Just then, a small
mouse came there and started running up and down the lion's body. Because of this, the lion's sleep
was disturbed. The lion became angry and caught the mouse in his paw. The mouse, terrified, said,
'O King of the Forest! Please spare

In [3]:
def is_new_block(line):
    return bool(re.match(r'^[०-९0-9]+\.', line.strip()))


doc = pymupdf.open(file_path)

passage_blocks = []
current_block = []

for page in doc:
    text = page.get_text("text")  # 👈 NO OCR

    lines = text.splitlines()

    for line in lines:
        line = line.strip()

        if not line:
            continue

        # detect new story
        if is_new_block(line):
            if current_block:
                passage_blocks.append(" ".join(current_block))
                current_block = []

        current_block.append(line)

# last block
if current_block:
    passage_blocks.append(" ".join(current_block))

In [4]:
print(passage_blocks)

["१. ɭसिंहः च मूषकः एकɧ×स्मिन् वने एकः बलवान् ɭसिंहः वसिɟति ×स्मि। एकदा सिः वृक्ष×य अधः सिुप्तिः आसिीति्। तिदा एकः लघुः स्मिूषकः तित्र आगत्य ɭसिंह×य शरीरे इति×तितिः अधावति्। तिेन ɭसिंह×य ɟनद्रा भग्ना अभवति्। ɭसिंहः क्रुद्धः भूत्वा स्मिूषकं ह×तिेन अगृह्णाति्। स्मिूषकः भीतिः भूत्वा अवदति्- 'भो वनराज! स्मिां स्मिुञ्चतिु। कदाɡचति् अहस्मि् अɟपि भवतिः सिाहाय्यं कɝरष्याɠस्मि।' ɭसिंहः हɡसित्वा तिं अस्मिुञ्चति्। गच्छɟति काले एकदा ɭसिंहः व्याध×य जाले बद्धः अभवति्। सिः उच्चैः अगजर्जति्। ɭसिंह×य गजर्जनं श्रुत्वा स्मिूषकः तित्र आगच्छति्। सिः ×वकैः तिीक्ष्णैः दन्तिैः जालं अकृन्तिति्। ɭसिंहः जालाति् स्मिुक्तिः अभवति्। सिः स्मिूषक×य धन्यवादं अकरोति्।", "1. The Lion and the Mouse In a forest, there lived a powerful lion. One day, he was sleeping under a tree. Just then, a small mouse came there and started running up and down the lion's body. Because of this, the lion's sleep was disturbed. The lion became angry and caught the mouse in his paw. The mouse, terrified, said, 'O King of the Forest! Please 

In [12]:
import fitz as pymupdf
import os
import re
import unicodedata


# -----------------------------
# TITLE DETECTION
# -----------------------------
def is_title(line):
    return bool(re.match(r'^[०-९0-9]+\.\s*', line.strip()))


# -----------------------------
# CLEAN
# -----------------------------
def clean(line):
    line = unicodedata.normalize("NFKC", line)
    line = line.replace("\u200c", "").replace("\u200d", "")
    return re.sub(r'\s+', ' ', line).strip()


# -----------------------------
# LANGUAGE SCORE (ROBUST)
# -----------------------------
def lang_score(text):
    if not text:
        return "noise"

    dev = len(re.findall(r'[\u0900-\u097F]', text))
    eng = len(re.findall(r'[a-zA-Z]', text))
    total = len(text)

    if total == 0:
        return "noise"

    dev_ratio = dev / total
    eng_ratio = eng / total

    # strong Sanskrit
    if dev_ratio > 0.30:
        return "sanskrit"

    # strong English
    if eng_ratio > 0.40:
        return "english"

    return "noise"


# -----------------------------
# MAIN EXTRACTION
# -----------------------------
def extract_bilingual_passages(file_path):

    doc = pymupdf.open(file_path)

    passages = []

    current_title = None
    sanskrit = []
    english = []

    for page in doc:
        text = page.get_text("text")
        lines = text.splitlines()

        for raw in lines:
            line = clean(raw)
            if not line:
                continue

            # NEW TITLE
            if is_title(line):
                if current_title:
                    passages.append({
                        "title": current_title,
                        "sanskrit": " ".join(sanskrit).strip(),
                        "english": " ".join(english).strip()
                    })

                current_title = line
                sanskrit = []
                english = []
                continue

            # classify EVERY LINE independently
            label = lang_score(line)

            if label == "sanskrit":
                sanskrit.append(line)

            elif label == "english":
                english.append(line)

            else:
                # noise handling: attach to longer context intelligently
                if len(sanskrit) >= len(english):
                    sanskrit.append(line)
                else:
                    english.append(line)

    # last block
    if current_title:
        passages.append({
            "title": current_title,
            "sanskrit": " ".join(sanskrit).strip(),
            "english": " ".join(english).strip()
        })

    return passages


# -----------------------------
# RUN
# -----------------------------
file_path = os.path.join("..", "data", "sanskrit_english_bilingual_stories.pdf")

data = extract_bilingual_passages(file_path)

print(data)

[{'title': '१. ɭसिंहः च मूषकः', 'sanskrit': "एकɧ×स्मिन् वने एकः बलवान् ɭसिंहः वसिɟति ×स्मि। एकदा सिः वृक्ष×य अधः सिुप्तिः आसिीति्। तिदा एकः लघुः स्मिूषकः तित्र आगत्य ɭसिंह×य शरीरे इति×तितिः अधावति्। तिेन ɭसिंह×य ɟनद्रा भग्ना अभवति्। ɭसिंहः क्रुद्धः भूत्वा स्मिूषकं ह×तिेन अगृह्णाति्। स्मिूषकः भीतिः भूत्वा अवदति्- 'भो वनराज! स्मिां स्मिुञ्चतिु। कदाɡचति् अहस्मि् अɟपि भवतिः सिाहाय्यं कɝरष्याɠस्मि।' ɭसिंहः हɡसित्वा तिं अस्मिुञ्चति्। गच्छɟति काले एकदा ɭसिंहः व्याध×य जाले बद्धः अभवति्। सिः उच्चैः अगजर्जति्। ɭसिंह×य गजर्जनं श्रुत्वा स्मिूषकः तित्र आगच्छति्। सिः ×वकैः तिीक्ष्णैः दन्तिैः जालं अकृन्तिति्। ɭसिंहः जालाति् स्मिुक्तिः अभवति्। सिः स्मिूषक×य धन्यवादं अकरोति्।", 'english': ''}, {'title': '1. The Lion and the Mouse', 'sanskrit': '', 'english': "In a forest, there lived a powerful lion. One day, he was sleeping under a tree. Just then, a small mouse came there and started running up and down the lion's body. Because of this, the lion's sleep was disturbed. The lion became angry and caught

In [14]:
import fitz as pymupdf
import os
import re
import unicodedata


# =========================================================
# CONFIG
# =========================================================
DEVANAGARI_RE = re.compile(r'[\u0900-\u097F]')
LATIN_RE = re.compile(r'[a-zA-Z]')

TITLE_RE = re.compile(r'^[०-९0-9]+\.\s*|^[0-9]+\.\s*')


# =========================================================
# CLEANING
# =========================================================
def clean(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u200c", "").replace("\u200d", "")
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# =========================================================
# LANGUAGE SCORING (more stable than boolean rules)
# =========================================================
def language_score(text):
    if not text:
        return 0, 0

    d = len(DEVANAGARI_RE.findall(text))
    l = len(LATIN_RE.findall(text))
    t = len(text)

    return d / t, l / t


def is_title(line):
    return bool(TITLE_RE.match(line.strip()))


# =========================================================
# BLOCK DATA STRUCTURE
# =========================================================
class Block:
    def __init__(self, title):
        self.title = title
        self.sanskrit_lines = []
        self.english_lines = []

    def add(self, line):
        d_ratio, l_ratio = language_score(line)

        if d_ratio > 0.2:
            self.sanskrit_lines.append(line)
        elif l_ratio > 0.3:
            self.english_lines.append(line)
        else:
            # fallback: attach to Sanskrit by default
            self.sanskrit_lines.append(line)

    def finalize(self):
        return {
            "title": self.title,
            "sanskrit": " ".join(self.sanskrit_lines).strip(),
            "english": " ".join(self.english_lines).strip()
        }


# =========================================================
# MAIN PIPELINE (NO OCR FIRST)
# =========================================================
def extract_blocks_from_pdf(file_path):

    doc = pymupdf.open(file_path)

    blocks = []
    current = None

    for page in doc:
        text = page.get_text("text")  # IMPORTANT: no OCR
        lines = text.splitlines()

        for raw in lines:
            line = clean(raw)
            if not line:
                continue

            # NEW TITLE
            if is_title(line):
                if current:
                    blocks.append(current.finalize())

                current = Block(title=line)
                continue

            if current:
                current.add(line)

    if current:
        blocks.append(current.finalize())

    return blocks


# =========================================================
# OPTIONAL OCR FALLBACK (ONLY FOR BAD BLOCKS)
# =========================================================
def needs_ocr(block):
    """
    Detect corrupted OCR-text-layer output.
    Example: mixed glyphs like 'ɭसिंहः'
    """
    text = block["sanskrit"] + block["english"]

    # too many weird symbols = bad extraction
    weird_chars = len(re.findall(r'[^\u0900-\u097Fa-zA-Z0-9\s\.,\'"।-]', text))
    total = len(text)

    if total == 0:
        return True

    return (weird_chars / total) > 0.15


def refine_with_ocr(page_text_block):
    """
    Placeholder:
    You would run OCR ONLY on specific page regions here.
    (NOT full page blindly)
    """
    # integrate pytesseract or easyocr if needed
    return page_text_block


# =========================================================
# PIPELINE WRAPPER
# =========================================================
def extract_bilingual_passages(file_path, enable_ocr=False):

    blocks = extract_blocks_from_pdf(file_path)

    if not enable_ocr:
        return blocks

    # optional second pass correction
    refined = []

    for b in blocks:
        if needs_ocr(b):
            # placeholder hook
            b = refine_with_ocr(b)

        refined.append(b)

    return refined


# =========================================================
# RUN
# =========================================================
file_path = os.path.join("..", "data", "sanskrit_english_bilingual_stories.pdf")

data = extract_bilingual_passages(file_path, enable_ocr=False)

print(data)

[{'title': '१. ɭसिंहः च मूषकः', 'sanskrit': "एकɧ×स्मिन् वने एकः बलवान् ɭसिंहः वसिɟति ×स्मि। एकदा सिः वृक्ष×य अधः सिुप्तिः आसिीति्। तिदा एकः लघुः स्मिूषकः तित्र आगत्य ɭसिंह×य शरीरे इति×तितिः अधावति्। तिेन ɭसिंह×य ɟनद्रा भग्ना अभवति्। ɭसिंहः क्रुद्धः भूत्वा स्मिूषकं ह×तिेन अगृह्णाति्। स्मिूषकः भीतिः भूत्वा अवदति्- 'भो वनराज! स्मिां स्मिुञ्चतिु। कदाɡचति् अहस्मि् अɟपि भवतिः सिाहाय्यं कɝरष्याɠस्मि।' ɭसिंहः हɡसित्वा तिं अस्मिुञ्चति्। गच्छɟति काले एकदा ɭसिंहः व्याध×य जाले बद्धः अभवति्। सिः उच्चैः अगजर्जति्। ɭसिंह×य गजर्जनं श्रुत्वा स्मिूषकः तित्र आगच्छति्। सिः ×वकैः तिीक्ष्णैः दन्तिैः जालं अकृन्तिति्। ɭसिंहः जालाति् स्मिुक्तिः अभवति्। सिः स्मिूषक×य धन्यवादं अकरोति्।", 'english': ''}, {'title': '1. The Lion and the Mouse', 'sanskrit': '', 'english': "In a forest, there lived a powerful lion. One day, he was sleeping under a tree. Just then, a small mouse came there and started running up and down the lion's body. Because of this, the lion's sleep was disturbed. The lion became angry and caught

In [16]:
import fitz as pymupdf
import os
import re
import unicodedata


# =========================================================
# CONFIG
# =========================================================
TITLE_RE = re.compile(r'^[०-९0-9]+\.\s*|^[0-9]+\.\s*')
DEVANAGARI_RE = re.compile(r'[\u0900-\u097F]')
LATIN_RE = re.compile(r'[a-zA-Z]')


# =========================================================
# CLEAN
# =========================================================
def clean(text):
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u200c", "").replace("\u200d", "")
    return re.sub(r'\s+', ' ', text).strip()


# =========================================================
# DETECT TITLE
# =========================================================
def is_title(line):
    return bool(TITLE_RE.match(line))


# =========================================================
# LANGUAGE TYPE (STRICT)
# =========================================================
def detect_language(text):
    d = len(DEVANAGARI_RE.findall(text))
    l = len(LATIN_RE.findall(text))

    if d > l:
        return "sanskrit"
    elif l > d:
        return "english"
    else:
        return "unknown"


# =========================================================
# CORE PIPELINE
# =========================================================
def extract_bilingual(file_path):

    doc = pymupdf.open(file_path)

    stories = []

    current_title = None
    current_sanskrit = []
    current_english = []

    # buffer to collect raw paragraphs before pairing
    buffer = []

    def flush_buffer():
        """
        Convert sequential paragraphs into structured bilingual pairing
        """
        nonlocal current_sanskrit, current_english

        mode = None

        for para in buffer:
            lang = detect_language(para)

            # English paragraph
            if lang == "english":
                current_english.append(para)

            # Sanskrit paragraph
            elif lang == "sanskrit":
                current_sanskrit.append(para)

            # fallback → attach to last known
            else:
                if current_sanskrit and not current_english:
                    current_sanskrit.append(para)
                else:
                    current_english.append(para)

        buffer.clear()

    for page in doc:
        text = page.get_text("text")
        lines = [clean(l) for l in text.splitlines() if clean(l)]

        for line in lines:

            # NEW STORY
            if is_title(line):

                # flush previous story
                if current_title:
                    flush_buffer()

                    stories.append({
                        "title": current_title,
                        "sanskrit": " ".join(current_sanskrit).strip(),
                        "english": " ".join(current_english).strip()
                    })

                    current_sanskrit = []
                    current_english = []
                    buffer = []

                current_title = line
                continue

            buffer.append(line)

    # FINAL FLUSH
    if current_title:
        flush_buffer()

        stories.append({
            "title": current_title,
            "sanskrit": " ".join(current_sanskrit).strip(),
            "english": " ".join(current_english).strip()
        })

    return stories


# =========================================================
# RUN
# =========================================================
file_path = os.path.join("..", "data", "sanskrit_english_bilingual_stories.pdf")

data = extract_bilingual(file_path)

print(data)

[{'title': '१. ɭसिंहः च मूषकः', 'sanskrit': "एकɧ×स्मिन् वने एकः बलवान् ɭसिंहः वसिɟति ×स्मि। एकदा सिः वृक्ष×य अधः सिुप्तिः आसिीति्। तिदा एकः लघुः स्मिूषकः तित्र आगत्य ɭसिंह×य शरीरे इति×तितिः अधावति्। तिेन ɭसिंह×य ɟनद्रा भग्ना अभवति्। ɭसिंहः क्रुद्धः भूत्वा स्मिूषकं ह×तिेन अगृह्णाति्। स्मिूषकः भीतिः भूत्वा अवदति्- 'भो वनराज! स्मिां स्मिुञ्चतिु। कदाɡचति् अहस्मि् अɟपि भवतिः सिाहाय्यं कɝरष्याɠस्मि।' ɭसिंहः हɡसित्वा तिं अस्मिुञ्चति्। गच्छɟति काले एकदा ɭसिंहः व्याध×य जाले बद्धः अभवति्। सिः उच्चैः अगजर्जति्। ɭसिंह×य गजर्जनं श्रुत्वा स्मिूषकः तित्र आगच्छति्। सिः ×वकैः तिीक्ष्णैः दन्तिैः जालं अकृन्तिति्। ɭसिंहः जालाति् स्मिुक्तिः अभवति्। सिः स्मिूषक×य धन्यवादं अकरोति्।", 'english': ''}, {'title': '1. The Lion and the Mouse', 'sanskrit': '', 'english': "In a forest, there lived a powerful lion. One day, he was sleeping under a tree. Just then, a small mouse came there and started running up and down the lion's body. Because of this, the lion's sleep was disturbed. The lion became angry and caught